In [1]:
# 1
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [2]:
# 2
fake = Faker()
num_customers = 150
transactions_per_customer = 150
customers = []
for _ in range(num_customers):
    first_name = fake.first_name()
    last_name = fake.last_name()
    customer_id = fake.unique.uuid4()  # Unique customer ID
    r = random.random()
    if r < 0.33:
        customer_type = "frequent"
    elif r < 0.66:
        customer_type = "regular"
    else:
        customer_type = "rare"
    customers.append([first_name, last_name, customer_id, customer_type])

In [3]:
# 3
df_customers = pd.DataFrame(
    customers,
    columns=['first_name', 'last_name', 'customer_id', 'customer_type']
)
transactions = []
vendors = ['Walmart', 'Amazon', 'Costco', 'Starbucks', 'American Airlines','United Airlines','CVS Pharmacy','Delta Airlines','Subway','Chick-Fil-A',
           'Chipotle','Panera Bread','Macdonalds']

# Vendor list (for retail, food chains, airlines, and pharmacy)
retail_and_airlines_vendors = ['Walmart', 'Amazon', 'Costco', 'American Airlines', 'Delta Airlines', 'United Airlines']
food_chains = ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A']
pharmacy = ['CVS Pharmacy']

vendor_groups = {
    0: ['American Airlines', 'Walmart'],  # First 30 customers
    1: ['Delta Airlines', 'Costco'],  # Next 30 customers
    2: ['United Airlines', 'Amazon'],  # Next 30 customers
    3: ['American Airlines', 'Costco'],  # Next 30 customers
    4: ['Walmart', 'Delta Airlines'],  # Last 30 customers
}

In [4]:
# 4
print(df_customers.head())
print(df_customers.columns)
print(df_customers['customer_type'].value_counts())

  first_name last_name                           customer_id customer_type
0       Chad      Frey  08ce0474-098e-4dd2-8846-3e0308ab2d8d          rare
1      Jesus   Webster  ae514cf3-79a2-4881-b176-bbbd013338ef          rare
2    Crystal     James  0d143c39-935b-43f0-9dd1-e9aa81514122          rare
3  Alexander     Allen  cf443ebc-b808-4b5d-a7ca-f9ca293908a5          rare
4     Nicole  Richmond  cbef657e-6c9d-450b-8f45-b5bfe784ee1b      frequent
Index(['first_name', 'last_name', 'customer_id', 'customer_type'], dtype='str')
customer_type
frequent    52
regular     52
rare        46
Name: count, dtype: int64


In [5]:
# 5

# Function to generate transaction amounts based on vendor
def generate_transaction_amount(vendor):
    if vendor in ['American Airlines', 'United Airlines', 'Delta Airlines']:
        return round(random.uniform(100.0, 500.0), 2)  # Airline vendors between $100 and $500
    elif vendor in ['Walmart', 'Costco', 'Amazon']:
        return round(random.uniform(50.0, 500.0), 2)  # Retail vendors between $50 and $500
    elif vendor in ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A']:
        return round(random.uniform(5.0, 100.0), 2)  # Food chains between $5 and $100
    elif vendor == 'CVS Pharmacy':
        return round(random.uniform(10.0, 100.0), 2)  # CVS Pharmacy between $10 and $100
    else:
        return round(random.uniform(80.0, 500.0), 2)  # Other vendors between $80 and $500


for idx, customer in df_customers.iterrows():
    first_name = customer['first_name']
    last_name = customer['last_name']
    customer_id = customer['customer_id']
    customer_type = customer['customer_type']

    # Random starting date for the first transaction
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2025, 12, 31)

    # Determine primary and secondary vendors for the customer based on their index
    primary_vendors = vendor_groups[idx // 30]  # Determine primary vendors based on index
    secondary_vendors = [vendor for vendor in vendors if vendor not in primary_vendors]

    # Create a transaction list for each customer
    all_vendors = primary_vendors + secondary_vendors

    for i in range(transactions_per_customer):
        # Select a vendor based on primary and secondary
        vendor = random.choice(all_vendors)

        # Base days between transactions by customer_type: frequent 3-7, regular 8-15, rare 16-30
        if customer_type == 'frequent':
            base_days = random.randint(3, 7)
        elif customer_type == 'regular':
            base_days = random.randint(8, 15)
        else:
            base_days = random.randint(16, 30)

        # Vendor adjustment
        if vendor in ['American Airlines', 'United Airlines', 'Delta Airlines']:
            base_days += random.randint(5, 10)
        elif vendor in ['Walmart', 'Amazon', 'Costco']:
            base_days += random.randint(-3, 3)
        elif vendor in ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A', 'Macdonalds']:
            base_days += random.randint(-2, 2)
        elif vendor == 'CVS Pharmacy':
            base_days += random.randint(-5, 5)

        # Add noise
        base_days += random.randint(-2, 2)
        days_diff = max(0, base_days)

        transaction_date = start_date + timedelta(days=i * days_diff)
        if transaction_date > end_date:
            break

        # Generate a random transaction amount based on the vendor
        amount = generate_transaction_amount(vendor)

        # Randomly assign "Debit" or "Credit" transaction type
        transaction_type = random.choice(['Debit', 'Credit'])

        # Add transaction to the list
        transactions.append([first_name, last_name, customer_id, fake.uuid4(), transaction_date, amount, vendor, transaction_type])

# Create DataFrame from the transactions list
df_transactions = pd.DataFrame(transactions, columns=['first_name', 'last_name', 'customer_id', 'transaction_id', 'transaction_datetime', 
                                                      'amount', 'vendor', 'transaction_type'])

# View the first few transactions to verify vendor distribution
print(df_transactions[['customer_id', 'vendor', 'transaction_datetime']].head(20))





                             customer_id             vendor  \
0   08ce0474-098e-4dd2-8846-3e0308ab2d8d    United Airlines   
1   08ce0474-098e-4dd2-8846-3e0308ab2d8d       CVS Pharmacy   
2   08ce0474-098e-4dd2-8846-3e0308ab2d8d          Starbucks   
3   08ce0474-098e-4dd2-8846-3e0308ab2d8d     Delta Airlines   
4   08ce0474-098e-4dd2-8846-3e0308ab2d8d            Walmart   
5   08ce0474-098e-4dd2-8846-3e0308ab2d8d    United Airlines   
6   08ce0474-098e-4dd2-8846-3e0308ab2d8d             Amazon   
7   08ce0474-098e-4dd2-8846-3e0308ab2d8d           Chipotle   
8   08ce0474-098e-4dd2-8846-3e0308ab2d8d    United Airlines   
9   08ce0474-098e-4dd2-8846-3e0308ab2d8d        Chick-Fil-A   
10  08ce0474-098e-4dd2-8846-3e0308ab2d8d             Costco   
11  08ce0474-098e-4dd2-8846-3e0308ab2d8d    United Airlines   
12  08ce0474-098e-4dd2-8846-3e0308ab2d8d  American Airlines   
13  08ce0474-098e-4dd2-8846-3e0308ab2d8d             Subway   
14  08ce0474-098e-4dd2-8846-3e0308ab2d8d         Macdon

In [6]:
# 7
# Recompute next transaction date
df_transactions = df_transactions.sort_values(
    ['customer_id', 'transaction_datetime']
)

df_transactions['next_transaction_date'] = (
    df_transactions.groupby(['customer_id'])['transaction_datetime']
    .shift(-1)
)

df_transactions['likelihood_prediction'] = (
    df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']
).dt.days

# Drop last transactions (no next date)
df_transactions = df_transactions.dropna(subset=['next_transaction_date'])

# Basic checks
print("Shape:", df_transactions.shape)
print("Date range:", df_transactions['transaction_datetime'].min(), "to", df_transactions['transaction_datetime'].max())
print("Min days:", df_transactions['likelihood_prediction'].min())
print("Max days:", df_transactions['likelihood_prediction'].max())
print("Mean days:", df_transactions['likelihood_prediction'].mean())

Shape: (6090, 10)
Date range: 2024-01-01 00:00:00 to 2025-12-27 00:00:00
Min days: 0.0
Max days: 216.0
Mean days: 16.071100164203614


In [7]:
# 8
# Number of transactions per customer
df_transactions['customer_transaction_count'] = (
    df_transactions.groupby('customer_id')['transaction_id']
    .transform('count')
)

print(df_transactions[['customer_id', 'customer_transaction_count']].head(10))

                               customer_id  customer_transaction_count
4233  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4234  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4235  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4241  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4249  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4239  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4238  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4246  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4250  0203ea34-e5ac-4a30-928f-88349466ef84                          50
4236  0203ea34-e5ac-4a30-928f-88349466ef84                          50


In [8]:
# 9
# How many times a customer purchased from a vendor
df_transactions['customer_vendor_txn_count'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_id']
    .transform('count')
)

print(df_transactions[['customer_id', 'vendor', 'customer_vendor_txn_count']].head(10))

                               customer_id             vendor  \
4233  0203ea34-e5ac-4a30-928f-88349466ef84       CVS Pharmacy   
4234  0203ea34-e5ac-4a30-928f-88349466ef84     Delta Airlines   
4235  0203ea34-e5ac-4a30-928f-88349466ef84  American Airlines   
4241  0203ea34-e5ac-4a30-928f-88349466ef84        Chick-Fil-A   
4249  0203ea34-e5ac-4a30-928f-88349466ef84             Amazon   
4239  0203ea34-e5ac-4a30-928f-88349466ef84             Subway   
4238  0203ea34-e5ac-4a30-928f-88349466ef84          Starbucks   
4246  0203ea34-e5ac-4a30-928f-88349466ef84            Walmart   
4250  0203ea34-e5ac-4a30-928f-88349466ef84         Macdonalds   
4236  0203ea34-e5ac-4a30-928f-88349466ef84    United Airlines   

      customer_vendor_txn_count  
4233                          6  
4234                          4  
4235                          2  
4241                          7  
4249                          4  
4239                          3  
4238                          5  
4246        

In [9]:
# 10
# Days since customer's first transaction
df_transactions['days_since_first_purchase'] = (
    df_transactions['transaction_datetime'] -
    df_transactions.groupby('customer_id')['transaction_datetime'].transform('min')
).dt.days

print(df_transactions[['customer_id', 'days_since_first_purchase']].head(10))

                               customer_id  days_since_first_purchase
4233  0203ea34-e5ac-4a30-928f-88349466ef84                          0
4234  0203ea34-e5ac-4a30-928f-88349466ef84                         10
4235  0203ea34-e5ac-4a30-928f-88349466ef84                         16
4241  0203ea34-e5ac-4a30-928f-88349466ef84                         16
4249  0203ea34-e5ac-4a30-928f-88349466ef84                         16
4239  0203ea34-e5ac-4a30-928f-88349466ef84                         18
4238  0203ea34-e5ac-4a30-928f-88349466ef84                         20
4246  0203ea34-e5ac-4a30-928f-88349466ef84                         26
4250  0203ea34-e5ac-4a30-928f-88349466ef84                         34
4236  0203ea34-e5ac-4a30-928f-88349466ef84                         39


In [10]:
# 11
# Month and weekday features
df_transactions['transaction_month'] = df_transactions['transaction_datetime'].dt.month
df_transactions['transaction_weekday'] = df_transactions['transaction_datetime'].dt.weekday

print(df_transactions[['transaction_datetime', 'transaction_month', 'transaction_weekday']].head(10))

     transaction_datetime  transaction_month  transaction_weekday
4233           2024-01-01                  1                    0
4234           2024-01-11                  1                    3
4235           2024-01-17                  1                    2
4241           2024-01-17                  1                    2
4249           2024-01-17                  1                    2
4239           2024-01-19                  1                    4
4238           2024-01-21                  1                    6
4246           2024-01-27                  1                    5
4250           2024-02-04                  2                    6
4236           2024-02-09                  2                    4


In [11]:
# 12
# Average spending per customer
df_transactions['customer_avg_spend'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform('mean')
)

print(df_transactions[['customer_id', 'amount', 'customer_avg_spend']].head(10))

                               customer_id  amount  customer_avg_spend
4233  0203ea34-e5ac-4a30-928f-88349466ef84   28.69            141.7818
4234  0203ea34-e5ac-4a30-928f-88349466ef84  325.28            141.7818
4235  0203ea34-e5ac-4a30-928f-88349466ef84  339.36            141.7818
4241  0203ea34-e5ac-4a30-928f-88349466ef84   50.27            141.7818
4249  0203ea34-e5ac-4a30-928f-88349466ef84  324.17            141.7818
4239  0203ea34-e5ac-4a30-928f-88349466ef84    7.87            141.7818
4238  0203ea34-e5ac-4a30-928f-88349466ef84   40.55            141.7818
4246  0203ea34-e5ac-4a30-928f-88349466ef84  251.47            141.7818
4250  0203ea34-e5ac-4a30-928f-88349466ef84  113.01            141.7818
4236  0203ea34-e5ac-4a30-928f-88349466ef84  332.93            141.7818


In [12]:
# 13
# Average time gap between purchases for each customer
df_transactions['customer_avg_gap'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform('mean')
)

print(df_transactions[['customer_id','likelihood_prediction','customer_avg_gap']].head(10))

                               customer_id  likelihood_prediction  \
4233  0203ea34-e5ac-4a30-928f-88349466ef84                   10.0   
4234  0203ea34-e5ac-4a30-928f-88349466ef84                    6.0   
4235  0203ea34-e5ac-4a30-928f-88349466ef84                    0.0   
4241  0203ea34-e5ac-4a30-928f-88349466ef84                    0.0   
4249  0203ea34-e5ac-4a30-928f-88349466ef84                    2.0   
4239  0203ea34-e5ac-4a30-928f-88349466ef84                    2.0   
4238  0203ea34-e5ac-4a30-928f-88349466ef84                    6.0   
4246  0203ea34-e5ac-4a30-928f-88349466ef84                    8.0   
4250  0203ea34-e5ac-4a30-928f-88349466ef84                    5.0   
4236  0203ea34-e5ac-4a30-928f-88349466ef84                    2.0   

      customer_avg_gap  
4233             11.76  
4234             11.76  
4235             11.76  
4241             11.76  
4249             11.76  
4239             11.76  
4238             11.76  
4246             11.76  
4250           

In [13]:
# 14
# Remove rows where likelihood_prediction is NaN
df_transactions = df_transactions.dropna(subset=['likelihood_prediction'])

print("Dataset shape after cleaning:", df_transactions.shape)

Dataset shape after cleaning: (6090, 17)


In [14]:
# 15
# Features for model
# vendor_* columns are added in PIPELINE 2/4 (below). If you run this cell before that,
# we use the base feature list only; re-run after PIPELINE 2/4 for the full set.
FEATURES_FULL = [
    'amount',
    'customer_transaction_count',
    'customer_vendor_txn_count',
    'days_since_first_purchase',
    'transaction_month',
    'transaction_weekday',
    'customer_avg_spend',
    'customer_avg_gap',
    'vendor_preferred_day',
    'vendor_day_std',
    'vendor_preferred_month',
    'vendor_month_std',
    'vendor_monthly_frequency',
]
FEATURES_BASE = [
    'amount',
    'customer_transaction_count',
    'customer_vendor_txn_count',
    'days_since_first_purchase',
    'transaction_month',
    'transaction_weekday',
    'customer_avg_spend',
    'customer_avg_gap',
]

if 'vendor_preferred_day' in df_transactions.columns:
    features = FEATURES_FULL
else:
    features = FEATURES_BASE
    print(
        "Note: vendor pattern columns are not in df_transactions yet. "
        "Run PIPELINE 1/4 and 2/4 below, then re-run this cell to include them."
    )

X = df_transactions[features]
y = df_transactions['likelihood_prediction']

print(X.head())
print("Target sample:")
print(y.head())

In [ ]:
# 16
print("Shape:", df_transactions.shape)
print("Date range:", 
      df_transactions['transaction_datetime'].min(), 
      "to", 
      df_transactions['transaction_datetime'].max())

print(df_transactions['likelihood_prediction'].describe())

Shape: (6051, 17)
Date range: 2024-01-01 00:00:00 to 2025-12-05 00:00:00
count    6051.000000
mean       15.982317
std        21.443206
min         0.000000
25%         4.000000
50%         9.000000
75%        20.000000
max       292.000000
Name: likelihood_prediction, dtype: float64


In [ ]:
# 17
print("Date range:", df_transactions['transaction_datetime'].min(), "to", df_transactions['transaction_datetime'].max())
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Any negative?", (df_transactions['likelihood_prediction'] < 0).any())

Date range: 2024-01-01 00:00:00 to 2025-12-05 00:00:00
Min likelihood: 0.0
Max likelihood: 292.0
Any negative? False


In [ ]:
# 18
print(df_transactions['likelihood_prediction'].describe())
print(df_transactions.groupby('customer_id')['likelihood_prediction'].mean().head())
print(df_transactions.groupby('vendor')['likelihood_prediction'].mean())

count    6051.000000
mean       15.982317
std        21.443206
min         0.000000
25%         4.000000
50%         9.000000
75%        20.000000
max       292.000000
Name: likelihood_prediction, dtype: float64
customer_id
06eb7127-8fe9-41f0-9bd4-5f5010846afd     9.622642
08918d79-029a-4db9-ba21-069623eb1c99    14.638298
0a8bb8fe-ce52-4007-930e-071b7be01ae1    17.500000
0b2e8ff5-0bf6-4a1a-8d50-466eac435e08    24.166667
0cacc7e8-5b44-41ef-88a7-8b76d1ddf3b5    12.350877
Name: likelihood_prediction, dtype: float64
vendor
Amazon               15.289941
American Airlines    20.579196
CVS Pharmacy         13.832311
Chick-Fil-A          15.097889
Chipotle             15.534934
Costco               14.616228
Delta Airlines       20.938503
Macdonalds           15.203704
Panera Bread         14.721248
Starbucks            13.919492
Subway               16.004149
United Airlines      19.597087
Walmart              14.655022
Name: likelihood_prediction, dtype: float64


In [ ]:
# 19
same_day_transactions = df_transactions[df_transactions.duplicated(
    subset=['customer_id', 'vendor', 'transaction_datetime'], keep=False
)]
print(same_day_transactions[['customer_id', 'vendor', 'transaction_datetime']].head(10))

                               customer_id           vendor  \
1711  08918d79-029a-4db9-ba21-069623eb1c99  United Airlines   
1721  08918d79-029a-4db9-ba21-069623eb1c99  United Airlines   
4964  0cacc7e8-5b44-41ef-88a7-8b76d1ddf3b5         Chipotle   
4992  0cacc7e8-5b44-41ef-88a7-8b76d1ddf3b5         Chipotle   
208   0fc0acfe-812d-4691-8229-c19f4dfa26c3          Walmart   
214   0fc0acfe-812d-4691-8229-c19f4dfa26c3          Walmart   
5159  2540188e-c865-4bcf-8134-70c456388ee2     Panera Bread   
5161  2540188e-c865-4bcf-8134-70c456388ee2     Panera Bread   
1569  268f85f1-e61b-4b90-a29f-85c3dd6c8eec     CVS Pharmacy   
1595  268f85f1-e61b-4b90-a29f-85c3dd6c8eec     CVS Pharmacy   

     transaction_datetime  
1711           2025-01-10  
1721           2025-01-10  
4964           2024-08-12  
4992           2024-08-12  
208            2024-03-01  
214            2024-03-01  
5159           2024-09-09  
5161           2024-09-09  
1569           2024-01-01  
1595           2024-01-01 

## Export pipeline (4 code cells below)

Order: **data generation** (above) → **PIPELINE 1/4** (sort + vendor-level likelihood) → **PIPELINE 2/4** (feature engineering) → **PIPELINE 3/4** (in-memory sanity) → **PIPELINE 4/4** (**export CSV — run last**).


In [ ]:
# PIPELINE 1/4 — Vendor-level likelihood (same customer + same vendor)
# Recompute likelihood at customer+vendor level (days until next purchase at same vendor)
df_transactions = df_transactions.sort_values(
    ['customer_id', 'vendor', 'transaction_datetime']
).reset_index(drop=True)

df_transactions['next_transaction_date'] = df_transactions.groupby(
    ['customer_id', 'vendor']
)['transaction_datetime'].shift(-1)

df_transactions['likelihood_prediction'] = (
    df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']
).dt.days

# Drop rows with no next transaction (last purchase per customer+vendor)
df_transactions = df_transactions.dropna(subset=['next_transaction_date'])

# Recompute customer_avg_gap on clean data
df_transactions['customer_avg_gap'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform('mean')
)

print("Shape:", df_transactions.shape)
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Any negative?", (df_transactions['likelihood_prediction'] < 0).any())
print(df_transactions['likelihood_prediction'].describe())


Shape: (4243, 17)
Min likelihood: 0.0
Max likelihood: 639.0
Any negative? False
count    4243.000000
mean       99.736036
std        97.620906
min         0.000000
25%        28.000000
50%        69.000000
75%       140.000000
max       639.000000
Name: likelihood_prediction, dtype: float64


In [ ]:
# PIPELINE 2/4 — Temporal + vendor features (expanded)
# Temporal features — what just happened (not all-time history)

df_transactions = df_transactions.sort_values(['customer_id', 'transaction_datetime']).reset_index(drop=True)

df_transactions['days_since_last_transaction'] = (
    df_transactions['transaction_datetime'] -
    df_transactions.groupby('customer_id')['transaction_datetime'].shift(1)
).dt.days

df_transactions['lag_1_gap'] = df_transactions.groupby('customer_id')['likelihood_prediction'].shift(1)
df_transactions['lag_2_gap'] = df_transactions.groupby('customer_id')['likelihood_prediction'].shift(2)
df_transactions['lag_3_gap'] = df_transactions.groupby('customer_id')['likelihood_prediction'].shift(3)

df_transactions['rolling_avg_gap_7'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=2).mean())
)
df_transactions['rolling_avg_gap_3'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
)
df_transactions['rolling_avg_gap_14'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform(lambda x: x.shift(1).rolling(14, min_periods=5).mean())
)

df_transactions['gap_trend_short'] = df_transactions['rolling_avg_gap_3'] - df_transactions['rolling_avg_gap_7']
df_transactions['gap_trend_long'] = df_transactions['rolling_avg_gap_7'] - df_transactions['rolling_avg_gap_14']

df_transactions['rolling_spend_3'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
)
df_transactions['rolling_spend_7'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=3).mean())
)
df_transactions['spend_trend'] = df_transactions['rolling_spend_3'] - df_transactions['rolling_spend_7']

vendor_avg_gap = df_transactions.groupby(['customer_id', 'vendor'])['likelihood_prediction'].transform('mean')
df_transactions['vendor_gap_vs_avg'] = vendor_avg_gap - df_transactions['customer_avg_gap']

vendor_avg_spend = df_transactions.groupby(['customer_id', 'vendor'])['amount'].transform('mean')
df_transactions['vendor_spend_vs_avg'] = vendor_avg_spend - df_transactions['customer_avg_spend']

vendor_category_map = {
    'American Airlines': 'airline', 'Delta Airlines': 'airline', 'United Airlines': 'airline',
    'Walmart': 'retail', 'Amazon': 'retail', 'Costco': 'retail',
    'Starbucks': 'food', 'Chipotle': 'food', 'Subway': 'food',
    'Chick-Fil-A': 'food', 'Panera Bread': 'food', 'Macdonalds': 'food',
    'CVS Pharmacy': 'pharmacy'
}
cat_map = {'food': 0, 'pharmacy': 1, 'retail': 2, 'airline': 3, 'other': 4}
df_transactions['vendor_category'] = df_transactions['vendor'].map(vendor_category_map).fillna('other')
df_transactions['vendor_category_code'] = df_transactions['vendor_category'].map(cat_map)

# --- Purchase pattern features (day of month + seasonality) ---

# Average day-of-month this customer buys from this vendor
df_transactions['transaction_day'] = pd.to_datetime(
    df_transactions['transaction_datetime']
).dt.day

df_transactions['vendor_preferred_day'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_day']
    .transform('mean')
).round(0)

# How consistent is the day-of-month (low std = very regular buyer)
df_transactions['vendor_day_std'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_day']
    .transform('std')
).fillna(0)

# Which months does this customer buy from this vendor
df_transactions['vendor_preferred_month'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_month']
    .transform('mean')
).round(0)

# Month consistency
df_transactions['vendor_month_std'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_month']
    .transform('std')
).fillna(0)

# Purchase frequency per month (how many times per month on average)
df_transactions['vendor_monthly_frequency'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_id']
    .transform('count') /
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_month']
    .transform('nunique')
).round(2)

print("Pattern features added:")
print(df_transactions[[
    'vendor', 'vendor_preferred_day', 'vendor_day_std',
    'vendor_preferred_month', 'vendor_month_std', 'vendor_monthly_frequency'
]].head(10))

# dropna — require enough history for lags and rolling windows
df_transactions = df_transactions.dropna(subset=[
    'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap',
    'rolling_avg_gap_3', 'rolling_avg_gap_7', 'rolling_avg_gap_14',
    'rolling_spend_3', 'rolling_spend_7',
])

print("Shape after temporal features + dropna:", df_transactions.shape)
print(df_transactions[[
    'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap',
    'rolling_avg_gap_3', 'rolling_avg_gap_7', 'rolling_avg_gap_14',
    'gap_trend_short', 'gap_trend_long', 'rolling_spend_3', 'rolling_spend_7', 'spend_trend',
    'vendor_gap_vs_avg', 'vendor_spend_vs_avg', 'vendor_category_code',
]].describe())


Shape after temporal features + dropna: (3493, 33)
       days_since_last_transaction    lag_1_gap    lag_2_gap    lag_3_gap  \
count                  3493.000000  3493.000000  3493.000000  3493.000000   
mean                     17.435156    95.674778    97.389064    98.778414   
std                      25.753616    91.258612    93.616070    96.041942   
min                       0.000000     0.000000     0.000000     0.000000   
25%                       3.000000    28.000000    28.000000    28.000000   
50%                       8.000000    67.000000    68.000000    68.000000   
75%                      20.000000   135.000000   138.000000   140.000000   
max                     276.000000   536.000000   587.000000   587.000000   

       rolling_avg_gap_3  rolling_avg_gap_7  rolling_avg_gap_14  \
count        3493.000000        3493.000000         3493.000000   
mean           97.280752          99.544372          100.410927   
std            58.612933          48.185941           

In [ ]:
# PIPELINE 3/4 — In-memory sanity check (before export)
print("Ready to export — shape:", df_transactions.shape)
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Mean likelihood:", df_transactions['likelihood_prediction'].mean())
print("Columns:", list(df_transactions.columns))


Ready to export — shape: (3493, 33)
Min likelihood: 0.0
Max likelihood: 512.0
Mean likelihood: 93.87947323217864
Columns: ['first_name', 'last_name', 'customer_id', 'transaction_id', 'transaction_datetime', 'amount', 'vendor', 'transaction_type', 'next_transaction_date', 'likelihood_prediction', 'customer_transaction_count', 'customer_vendor_txn_count', 'days_since_first_purchase', 'transaction_month', 'transaction_weekday', 'customer_avg_spend', 'customer_avg_gap', 'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap', 'rolling_avg_gap_7', 'rolling_avg_gap_3', 'rolling_avg_gap_14', 'gap_trend_short', 'gap_trend_long', 'rolling_spend_3', 'rolling_spend_7', 'spend_trend', 'vendor_gap_vs_avg', 'vendor_spend_vs_avg', 'vendor_category', 'vendor_category_code']


In [ ]:
# PIPELINE 4/4 — Export CSV (last step; run after all feature engineering)
import os
expected = [
    'lag_3_gap', 'rolling_avg_gap_14', 'gap_trend_short', 'gap_trend_long',
    'spend_trend', 'rolling_spend_7', 'vendor_gap_vs_avg',
    'vendor_spend_vs_avg', 'vendor_category_code'
]
still_missing = [c for c in expected if c not in df_transactions.columns]
if still_missing:
    print("ERROR — still missing:", still_missing)
else:
    os.makedirs("predictor/data", exist_ok=True)
    df_transactions.to_csv("predictor/data/dataset1.csv", index=False)
    print("CSV exported successfully")
    print("Shape:", df_transactions.shape)
    print("Columns:", len(df_transactions.columns))


CSV exported successfully
Shape: (3493, 33)
Columns: 33
